# End-to-End ETL Pipeline

This notebook demonstrates a complete Extract-Transform-Load pipeline using the hybrid Rust+Python DataFrame engine.

## Scenario

We're processing e-commerce order data to create a regional sales summary report.

In [ ]:
# Setup
from pardox import DataFrame, read_csv
import time

print("ETL Pipeline Notebook")
print("=" * 40)

## Step 1: Generate Sample Data

First, we'll create a realistic dataset to work with.

In [ ]:
import csv
import random
from datetime import datetime, timedelta

def generate_orders(n_rows=1_000_000, output_path="orders_sample.csv"):
    """Generate synthetic e-commerce order data."""
    
    products = [f"SKU-{i:05d}" for i in range(100)]
    prices = [round(random.uniform(10, 500), 2) for _ in range(100)]
    customers = [f"CUST-{i:06d}" for i in range(10_000)]
    regions = ["North", "South", "East", "West", "Central"]
    
    start_date = datetime(2024, 1, 1)
    
    with open(output_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["order_id", "customer_id", "product_sku", 
                         "quantity", "unit_price", "order_date", "region"])
        
        for i in range(n_rows):
            order_id = f"ORD-{i:08d}"
            customer = random.choice(customers)
            sku_idx = random.randint(0, 99)
            product = products[sku_idx]
            quantity = random.randint(1, 10)
            price = prices[sku_idx]
            date = start_date + timedelta(days=random.randint(0, 365))
            region = random.choice(regions)
            
            writer.writerow([
                order_id, customer, product, quantity,
                price, date.strftime("%Y-%m-%d"), region
            ])
    
    return output_path

# Generate 1 million rows
print("Generating sample data...")
start = time.time()
csv_path = generate_orders(1_000_000)
print(f"Generated {csv_path} in {time.time() - start:.2f}s")

## Step 2: EXTRACT - Load the Data

In [ ]:
# Load CSV with Rust engine
print("Loading data...")
start = time.time()

df = read_csv(csv_path)

load_time = time.time() - start
print(f"Loaded {df.shape[0]:,} rows in {load_time:.2f}s")
print(f"Throughput: {df.shape[0] / load_time / 1_000_000:.1f}M rows/second")

In [ ]:
# Preview the data
df.head(10)

In [ ]:
# Check schema
print("Schema:")
for col in df.columns:
    print(f"  - {col}")

## Step 3: TRANSFORM - Clean and Enrich

In [ ]:
# Check for null values
print("Data quality check:")
print(f"Total rows: {df.shape[0]:,}")

In [ ]:
# Fill null values (if any)
df.fill_na("quantity", 1)  # Default to 1 item
df.fill_na("unit_price", 0.0)  # Free items
print("Null values handled")

In [ ]:
# Add derived columns
# Total value = quantity * unit_price
df["total_value"] = df["quantity"] * df["unit_price"]

# Extract month from date
df["order_month"] = df["order_date"].str.slice(0, 7)  # YYYY-MM

print("Added derived columns: total_value, order_month")
df[["order_date", "order_month", "quantity", "unit_price", "total_value"]].head(10)

In [ ]:
# Filter to high-value orders
print(f"Before filter: {df.shape[0]:,} rows")

df_high_value = df[df["total_value"] > 100]

print(f"After filter (total_value > $100): {df_high_value.shape[0]:,} rows")
print(f"Filtered out: {df.shape[0] - df_high_value.shape[0]:,} rows ({100 - df_high_value.shape[0]/df.shape[0]*100:.1f}%)")

## Step 4: LOAD - Aggregations and Output

In [ ]:
# Regional summary
print("=== REGIONAL SALES SUMMARY ===")
print()

regional = df_high_value.group_by("region").agg({
    "total_value": ["sum", "mean", "count"],
    "quantity": ["sum"]
})

print(regional)

In [ ]:
# Monthly trend
print("=== MONTHLY SALES TREND ===")
print()

monthly = df_high_value.group_by("order_month").agg({
    "total_value": "sum",
    "order_id": "count"
}).sort_by("order_month")

print(monthly)

In [ ]:
# Product analysis
print("=== TOP 10 PRODUCTS BY REVENUE ===")
print()

products = df_high_value.group_by("product_sku").agg({
    "total_value": "sum",
    "quantity": "sum"
}).sort_by("total_value", ascending=False)

products.head(10)

In [ ]:
# Customer analysis
print("=== TOP 10 CUSTOMERS BY SPENDING ===")
print()

customers = df_high_value.group_by("customer_id").agg({
    "total_value": "sum",
    "order_id": "count"
}).sort_by("total_value", ascending=False)

customers.head(10)

## Performance Summary

In [ ]:
# Full pipeline timing
print("=" * 50)
print("PIPELINE PERFORMANCE SUMMARY")
print("=" * 50)
print(f"Dataset size: {df.shape[0]:,} rows")
print(f"Columns: {len(df.columns)}")
print()
print(f"Load time: {load_time:.2f}s")
print(f"Throughput: {df.shape[0] / load_time / 1_000_000:.1f}M rows/second")
print()
print("Operations completed:")
print("  ✓ Data loading (CSV)")
print("  ✓ Null value handling")
print("  ✓ Derived column calculation")
print("  ✓ Filtering")
print("  ✓ GroupBy aggregations (4x)")
print("  ✓ Sorting")

## Cleanup

In [ ]:
# Remove generated file
import os
if os.path.exists(csv_path):
    os.remove(csv_path)
    print(f"Cleaned up: {csv_path}")

## Summary

This notebook demonstrated:
- Loading 1M rows from CSV in seconds
- Data cleaning with null handling
- Creating derived columns with vectorized operations
- Filtering with boolean masks
- Multiple GroupBy aggregations
- Sorting results

The entire pipeline processed 1 million rows interactively, demonstrating the performance benefits of the Rust-backed engine.